# 04 · Feature selection + train/test split
**Owner:** Wijayagurusinghe · **Branch:** `data/04-selection-split`

Drop near-duplicate features (|corr| ≥ 0.95, **fit on training cells only**), then split **by battery** (a cell is wholly in train or test). Save `train.parquet`, `test.parquet`, `feature_list.json`.


In [ ]:
# --- Colab setup (run first) ---
import os
REPO = '/content/ev-battery-rul-prediction'   # absolute path -> safe to re-run
if not os.path.exists(REPO):
    !git clone https://github.com/Chathumi-Senanayake/ev-battery-rul-prediction.git {REPO}
%cd {REPO}
!git pull -q
!pip install -q -r requirements.txt
import sys
if REPO not in sys.path: sys.path.insert(0, REPO)

# Try Google Drive (data persists across sessions). If auth fails, fall back
# to local Colab storage so you are never blocked.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/ev_battery_data'
except Exception as e:
    print('Drive mount failed -> using local storage:', e)
    DATA_DIR = '/content/ev_battery_data'
os.makedirs(DATA_DIR, exist_ok=True)
print('DATA_DIR =', DATA_DIR)


In [ ]:
import json, numpy as np, pandas as pd
data = pd.read_parquet(os.path.join(DATA_DIR,'features.parquet'))
feat_cols = json.load(open(os.path.join(DATA_DIR,'feature_list.json')))  # from nb 03
print('candidate features:', len(feat_cols))


### Battery-wise split, stratified by batch
Split by **cell** (a cell is wholly in train or test) AND keep each batch's share balanced — batch 2 cells are short-lived, so a plain random split could leave test over/under-represented. Test gets ~20% of each batch's cells.


In [ ]:
rng = np.random.default_rng(42)
test_cells = []
for b, grp in data.groupby('batch'):
    cells = np.array(grp['cell'].unique(), dtype=object); rng.shuffle(cells)
    k = max(1, round(0.2 * len(cells)))
    test_cells += list(cells[:k])
test  = data[data['cell'].isin(test_cells)].copy()
train = data[~data['cell'].isin(test_cells)].copy()
print('train cells:', train['cell'].nunique(), '| test cells:', test['cell'].nunique())
print('test cells by batch:', test.groupby('batch')['cell'].nunique().to_dict())


### Correlation filter — fit on TRAIN only


In [ ]:
corr = train[feat_cols].corr(method='spearman').abs()
upper = corr.where(np.triu(np.ones(corr.shape),k=1).astype(bool))
drop = [c for c in upper.columns if any(upper[c] >= 0.95)]
keep = [c for c in feat_cols if c not in drop]
print('dropped', len(drop), '-> kept', len(keep), 'features  (need > 20)')
json.dump(keep, open(os.path.join(DATA_DIR,'selected_features.json'),'w'), indent=2)


In [ ]:
cols = ['cell','batch','RUL'] + keep
train[cols].to_parquet(os.path.join(DATA_DIR,'train.parquet'), index=False)
test[cols].to_parquet(os.path.join(DATA_DIR,'test.parquet'), index=False)
print('saved train/test parquet')
